## Generate the .npz training set for Noise2Noise
#### Test-retest series with N >= 2 reliable examinations per eye

In [ ]:
import os
from pathlib import Path
DATA_ROOT = Path(os.environ.get("VF_DATA_ROOT", "/path/to/vf_oct_pairs"))   # see config/env.example.sh

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Paths
folder_raw = DATA_ROOT / "original_below350"
folder_denoised = DATA_ROOT / "denoised_gamma_corrected"

# Get a few filenames that exist in both
files = list(folder_raw.glob('*.npz'))[:5]

fig, axes = plt.subplots(len(files), 2, figsize=(10, 3*len(files)))

for i, f in enumerate(files):
    # Load Raw
    raw = np.load(f, allow_pickle=True)['td'].reshape(4, 13)
    
    # Load Denoised
    f_denoised = folder_denoised / f.name
    denoised = np.load(f_denoised, allow_pickle=True)['td'].reshape(4, 13)
    
    # Plot Raw
    axes[i, 0].imshow(raw, cmap='viridis', vmin=-35, vmax=10)
    axes[i, 0].set_title(f"Sample {i} RAW\nRange: [{raw.min():.1f}, {raw.max():.1f}]")
    
    # Plot Denoised
    axes[i, 1].imshow(denoised, cmap='viridis', vmin=-35, vmax=10)
    axes[i, 1].set_title(f"Sample {i} DENOISED\nRange: [{denoised.min():.1f}, {denoised.max():.1f}]")

plt.tight_layout()
plt.show()

### 1. Training dataset generation

In [ ]:
import sys
import os

from vf_dataset.config import train_boland, vffile, newoctfile
sys.path.append("../denoising/utils")
from DN_vf_tools import visualize, explore_df, plot_column_distribution, plot_vf, display_multiple_npz, plot_rnflt_and_vf, plot_longitudinal_vf
import pyreadr
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
vf24 = pyreadr.read_r(train_boland)
vf24_pd = vf24['bolandSS24.20.20.33']

In [ ]:
criteria = (vf24_pd['malfixrate'] <= 0.33) & (vf24_pd['falsenegrate'] <= 0.2) & (vf24_pd['falseposrate'] <= 0.2) & (vf24_pd['site'] != 5)
filtered_df = vf24_pd[criteria]
print(f"Before removal:{vf24_pd.shape[0]}   After removal: {filtered_df.shape[0]}" )

In [ ]:
vf24_pd = filtered_df
vf24_pd['test_id'] = vf24_pd['id'].astype(str) + '_' + \
                    vf24_pd['righteye'].astype(str) + '_' + \
                    vf24_pd['testdate'].astype(str)

In [ ]:
# Define dynamic and static columns for duplicate detection
td_cols = [f'td{i}' for i in range(1, 55) if i not in (26, 35)]
key_cols = ['id', 'righteye', 'age', 'duration'] + td_cols  # Columns used to define identical entries

# Find exact duplicates based on key columns
duplicates = filtered_df.duplicated(subset=key_cols, keep=False)
dupes_df = filtered_df[duplicates]

# Preview some duplicates (if any)
if len(dupes_df) > 0:
    display(dupes_df.sort_values(['id', 'righteye', 'age', 'duration']).head(5))

# Report number of duplicates found
n_dupes = len(dupes_df)
print(f"Found {n_dupes} duplicate entries based on {key_cols[:4]} + TDs")

# Remove exact duplicates
df_cleaned = filtered_df.drop_duplicates(subset=key_cols)
n_removed = filtered_df.shape[0] - df_cleaned.shape[0]
n_total = filtered_df.shape[0]
pct_removed = 100 * n_removed / n_total

print(f"Before removal:{filtered_df.shape[0]}   After removal: {df_cleaned.shape[0]} entries remain (removed {n_removed}, {pct_removed:.2f}% of total)")

In [ ]:
vffile = df_cleaned

In [ ]:
from collections import defaultdict

# 1. Number of tests per eye
counts = vffile.groupby('eyeid').size()

# 2. Keep only eyes with ≥ 2 tests in 90 DAYS window
eyes_with_enough_tests = counts[counts >= 2].index
df_multi = vffile[vffile['eyeid'].isin(eyes_with_enough_tests)].copy()

results = {}
valid_eyes_with_tests = {}
eyes_by_n_tests = defaultdict(list)

# For each eye with at least 2 tests
for eyeid, group in df_multi.groupby('eyeid'):
    group_sorted = group.sort_values('testdate')
    all_test_ids = group_sorted['test_id'].tolist()
    all_dates = group_sorted['testdate'].tolist()
    n_total = len(all_test_ids)

    found = False

    # Try longest possible valid sequence first
    for k in range(n_total, 1, -1):
        for i in range(n_total - k + 1):
            sub_dates = all_dates[i:i+k]
            duration = sub_dates[-1] - sub_dates[0]  # duration in days

            if duration <= 90:
                sub_ids = all_test_ids[i:i+k]
                valid_eyes_with_tests[eyeid] = sub_ids
                results[k] = results.get(k, 0) + 1
                eyes_by_n_tests[k].append(eyeid)
                found = True
                break
        if found:
            break

# Summary statistics
df_result = pd.Series(results).sort_index()

plt.figure(figsize=(8, 4))
df_result.plot(kind='bar')
plt.xlabel("Number of tests selected")
plt.ylabel("Number of eyes (first valid series ≤ 100 days)")
plt.title("Valid test series per eye (≤ 100-day span)")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
valid_eyes_with_tests

In [ ]:
keys_set = set(valid_eyes_with_tests.keys())
eyeids_in_df = set(vf24_pd['eyeid'])

covered = eyeids_in_df & keys_set
percentage = 100 * len(covered) / len(eyeids_in_df)

print(f"Covered: {len(covered)} / {len(eyeids_in_df)} eyes ({percentage:.2f}%)")


In [ ]:
def plot_longitudinal_vf_testdate(df, min_tests=5, max_eyes=5):
    import matplotlib.pyplot as plt
    import numpy as np

    td_cols = [f'td{i}' for i in range(1, 55) if i not in (26, 35)]

    # Filter eyes with enough tests
    eye_counts = df.groupby('eyeid').size()
    selected_eyes = eye_counts[eye_counts >= min_tests].index[:max_eyes]
    df_filtered = df[df['eyeid'].isin(selected_eyes)].copy()

    for eyeid in selected_eyes:
        eye_data = df_filtered[df_filtered['eyeid'] == eyeid].sort_values('testdate')
        patient_id = eye_data['id'].iloc[0]
        tds_list = []
        titles = []
        prev_testdate = None

        for i, (_, row) in enumerate(eye_data.iterrows()):
            tds = row[td_cols].values.astype(float)
            testdate = row['testdate']
            if prev_testdate is None:
                delta = 0
            else:
                delta = testdate - prev_testdate
            titles.append(f"{i+1}  Day {int(testdate)} ({int(delta)}d)")
            tds_list.append(tds)
            prev_testdate = testdate

        print(f"Patient ID: {patient_id} | EyeID: {eyeid} | {len(tds_list)} tests")
        visualize(tds_list, sizes=(2.2 * len(tds_list), 3), show_value=True, show_colorbar=True, title=titles)


In [ ]:
eyeid = 67.0    #replace with the eye identifier you want to visualize

# List of valid test_ids for this eye
valid_test_ids = valid_eyes_with_tests[eyeid]

# Strict filtering to keep only these tests
subset = vffile[vffile['test_id'].isin(valid_test_ids)]

# Call the plotting function on this subset
plot_longitudinal_vf_testdate(subset, min_tests=2, max_eyes=1)


In [ ]:
# 1. TD columns excluding blind spot locations
td_cols = [f'td{i}' for i in range(1, 55) if i not in (26, 35)]

# 2. Aggregate mean and variance by category (number of tests selected)
mean_by_category = {}
var_by_category = {}

for k, eyeids in eyes_by_n_tests.items():
    all_means = []
    all_vars = []

    for eyeid in eyeids:
        valid_test_ids = valid_eyes_with_tests[eyeid]
        subset = vffile[vffile['test_id'].isin(valid_test_ids)]
        td = subset[td_cols].astype(np.float32)

        mean_td = td.mean()   # mean per TD location (across tests)
        var_td  = td.var()    # variance per TD location (across tests)

        all_means.append(mean_td)
        all_vars.append(var_td)

    if all_means:
        # Average across eyes within this category
        mean_by_category[k] = pd.concat(all_means, axis=1).mean(axis=1)
        var_by_category[k]  = pd.concat(all_vars, axis=1).mean(axis=1)

# 3. Global mean (across all categories)
global_mean = pd.concat(mean_by_category.values(), axis=1).mean(axis=1)
global_var  = pd.concat(var_by_category.values(), axis=1).mean(axis=1)

# 4. Plot visual fields
plot_vf(global_mean, title="Visual Field — Mean of All Means")
plot_vf(global_var,  title="Visual Field — Mean of All Variances")


In [ ]:
import os
import numpy as np

N = 2  # minimum number of tests
td_cols = [f'td{i}' for i in range(1, 55) if i not in (26, 35)]

X_raw, Y_raw = [], []
id_arr, cid_arr = [], []
malfix_arr, fn_arr, fp_arr = [], [], []

for eyeid, test_ids in valid_eyes_with_tests.items():

    # Keep eyes with at least N tests
    if len(test_ids) < N:
        continue

    subset = vffile[vffile['test_id'].isin(test_ids)].sort_values('testdate')

    td_array = subset[td_cols].astype(np.float32).values  # (n_tests, 52)
    td_mean = td_array.mean(axis=0)  # (52,)

    for i in range(len(td_array)):
        row = subset.iloc[i]

        X_raw.append(td_array[i])
        Y_raw.append(td_mean)

        id_arr.append(row['id'])
        cid_arr.append(f"{row['id']}_{row['righteye']}_{row['testdate']}")
        malfix_arr.append(row['malfixrate'])
        fn_arr.append(row['falsenegrate'])
        fp_arr.append(row['falseposrate'])

# Convert to numpy arrays
X_raw = np.stack(X_raw)
Y_raw = np.stack(Y_raw)
id_arr = np.array(id_arr)
cid_arr = np.array(cid_arr)
malfix_arr = np.array(malfix_arr, dtype=np.float32)
fn_arr = np.array(fn_arr, dtype=np.float32)
fp_arr = np.array(fp_arr, dtype=np.float32)

# Save
np.savez_compressed(
    os.path.expandvars("$VF_DENOISE_ROOT/dataset_train_val_test/N2N/fine_tune_dataset_Nge2.npz"),
    td=X_raw, td_target=Y_raw,
    ids=id_arr, cids=cid_arr,
    malfix=malfix_arr, fnrate=fn_arr, fprate=fp_arr
)

print(f"Saved fine_tune_dataset_Nge2.npz with {len(X_raw)} samples.")


In [ ]:
'''
previously N=3 criteria:
Saved fine_tune_dataset_N3.npz with 2769 samples.
'''